In [ ]:
!pip install --upgrade hurst pandas_ta binance-historical-data plotly -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.1/115.1 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.8/272.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 31.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import pandas_ta as ta
import warnings
import pickle
import os
import time
import multiprocessing
from hurst import compute_Hc
from datetime import datetime, timedelta, date
from binance_historical_data import BinanceDataDumper
warnings.filterwarnings('ignore')

In [ ]:
def get_list_all_trading_pairs():
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency="1h",
    )
    return data_dumper.get_list_all_trading_pairs()

def filter_usdt_tickers(tickers):
    exclude_keywords = ["UPUSDT", "DOWNUSDT", "BEARUSDT", "BULLUSDT"]
    return [ticker for ticker in tickers if ticker.endswith("USDT") and not any(ex in ticker for ex in exclude_keywords)]

def find_first_data_date(ticker):
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency="1h",
    )
    return data_dumper.get_min_start_date_for_ticker(ticker)

def format_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

def detect_timestamp_unit(timestamp):
    num_digits = len(str(timestamp))
    if num_digits == 13:
        return 'ms'
    elif num_digits == 16:
        return 'us'
    else:
        raise ValueError(f"Timestamp không hợp lệ: {timestamp}")

def convert_timestamp(timestamp):
    unit = detect_timestamp_unit(timestamp)
    return pd.to_datetime(timestamp, unit=unit, errors='coerce')

def download_ticker(ticker, date_start, date_end, data_frequency="1h"):
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency= data_frequency,
    )
    date_start = datetime.strptime(date_start, "%Y-%m-%d").date()
    date_end = datetime.strptime(date_end, "%Y-%m-%d").date()
    data_dumper.dump_data(
        tickers = ticker,
        date_start = date_start,
        date_end = date_end,
        is_to_update_existing = False,
    )

def read_csv_file(file_path):
    df = pd.read_csv(file_path)
    df.columns = [
        "open_time", "Open", "High", "Low", "Close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
    ]
    df['open_time'] = df['open_time'].apply(convert_timestamp)
    df['close_time'] = df['close_time'].apply(convert_timestamp)
    return df

def get_csv_files(directory):
    try:
        if not os.path.exists(directory):
            print(f"Warning: Thư mục không tồn tại: {directory}")
            return []
        return [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith('.csv')]
    except Exception as e:
        print(f"Lỗi khi đọc thư mục {directory}: {str(e)}")
        return []

# Load data from file
def process_csv_files(ticker, data_frequency = "1h"):
    daily_path = os.path.join(os.getcwd(), f"spot/daily/klines/{ticker}/{data_frequency}")
    monthly_path = os.path.join(os.getcwd(), f"spot/monthly/klines/{ticker}/{data_frequency}")
    daily_files = get_csv_files(daily_path)
    monthly_files = get_csv_files(monthly_path)
    all_files = daily_files + monthly_files
    if not all_files:
        print(f"❗ Không có file CSV nào cho {ticker}")
        return None
    data = pd.concat([read_csv_file(file) for file in all_files], ignore_index=True)
    data.sort_values(by='open_time', inplace=True)
    return data

In [ ]:
ticker = 'BTCUSDT'
data_frequency = '15m'
download_ticker(ticker, "2017-03-01", "2025-03-01", data_frequency)
crypto_data = process_csv_files(ticker, data_frequency)

# Prepare data
data = crypto_data.reset_index(drop=True)
print(data)

---> Found overall tickers: 574
---> Filter to asked tickers: 7
------> Tickers left: 2
Download full data for 2 tickers: 
---> Data will be saved here: /content/spot
---> Data Frequency: 15m
---> Start Date: 20170301
---> End Date: 20250301


Tickers:   0%|          | 0/2 [00:00<?, ?it/s]

monthly files to download:   0%|          | 0/91 [00:00<?, ?files/s]

daily files to download:   0%|          | 0/1 [00:00<?, ?files/s]

monthly files to download: 0files [00:00, ?files/s]

daily files to download: 0files [00:00, ?files/s]

Tried to dump data for 2 tickers:
---> For BTCUSDT new data saved for: 91 months 1 days
---> For BTCUSD new data saved for: 0 months 0 days
                 open_time      Open      High       Low     Close  \
0      2017-08-17 04:15:00   4261.48   4270.41   4261.32   4261.45   
1      2017-08-17 04:30:00   4280.00   4310.07   4267.99   4310.07   
2      2017-08-17 04:45:00   4310.07   4313.62   4291.37   4308.83   
3      2017-08-17 05:00:00   4308.83   4328.69   4304.31   4304.31   
4      2017-08-17 05:15:00   4320.00   4320.00   4312.14   4320.00   
...                    ...       ...       ...       ...       ...   
263709 2025-03-01 22:45:00  86028.85  86262.74  85979.05  86116.35   
263710 2025-03-01 23:00:00  86116.36  86200.00  85810.44  85914.74   
263711 2025-03-01 23:15:00  85914.75  85914.75  85679.24  85679.25   
263712 2025-03-01 23:30:00  85679.25  85889.51  85679.24  85811.99   
263713 2025-03-01 23:45:00  85812.00  86100.00  85802.21  86064.53   

            volume 

In [ ]:
class Indicators:
    """
    Lớp Indicators tính toán các chỉ báo kỹ thuật từ dữ liệu giá.
    """
    def __init__(self, data):
        """
        Khởi tạo với DataFrame chứa các cột: 'Open', 'Close', 'High', 'Low'
        """
        self.data = data.copy()
        self.open = data['Open']
        self.close = data['Close']
        self.high = data['High']
        self.low = data['Low']
        self.returns = self.close.pct_change()

    @staticmethod
    def calculate_true_range(high, low, prev_close):
        """Tính True Range dựa vào high, low và giá đóng cửa của phiên trước."""
        high_low = high - low
        high_prev_close = np.abs(high - prev_close)
        low_prev_close = np.abs(low - prev_close)
        return np.maximum(high_low, np.maximum(high_prev_close, low_prev_close))

    def calculate_atr(self, period=14):
        """Tính Average True Range (ATR) dựa vào period."""
        prev_close = self.close.shift(1)  # Giá đóng cửa phiên trước
        true_range = self.calculate_true_range(self.high, self.low, prev_close)
        atr = ta.rma(true_range, period)
        return atr

    def compute_indicators(self):
        """Tính toán các chỉ báo cần thiết và lưu thành các thuộc tính nội bộ."""
        # Bollinger Bands: EMA 20 và độ lệch chuẩn
        self.bb = ta.ema(self.close, 20)
        self.bb_std = self.close.rolling(20).std()
        self.bbu = self.bb + 2 * self.bb_std  # Upper band
        self.bbl = self.bb - 2 * self.bb_std  # Lower band

        # RSI với chu kỳ 14
        self.rsi = ta.rsi(self.close, 14)

        # ATR tính từ True Range và sau đó áp dụng EMA
        self.atr = ta.ema(self.calculate_atr(period=14), 14)

        # Tính độ lệch chuẩn của giá sử dụng EWM
        self.std = self.close.ewm(span=14).std()

        # Tính giá trị trend: tỷ số ATR/Std rồi lấy trung bình trượt 3 kỳ
        self.trend_values = (self.atr / self.std).rolling(3).mean()

        # Điều kiện "neutral" dựa trên RSI
        self.neutral = (self.rsi >= 30) & (self.rsi <= 70)

        # Giới hạn của trend để xác định pha mean reversion
        self.trend_threshold = 0.6
        self.trend = self.trend_values <= self.trend_threshold

        # Tính rolling standard deviation từ returns, dùng để xác định mức SL và TP
        self.thresh_std = self.returns.ewm(span=14).std() * 4

class MainStrategy:
    """
    Lớp MainStrategy dùng các chỉ báo từ Indicators để chạy chiến lược, tính toán vị thế, và vẽ biểu đồ equity.
    """
    def __init__(self, indicators: Indicators):
        self.ind = indicators
        self.data = self.ind.data
        self.close = self.ind.close
        self.thresh_std = self.ind.thresh_std
        self.neutral = self.ind.neutral
        self.trend = self.ind.trend
        self.bbl = self.ind.bbl
        self.bbu = self.ind.bbu
        self.returns = self.ind.returns

        # Xác định các mức Take Profit và Stop Loss
        self.tp = self.thresh_std * 2
        self.sl = -self.thresh_std

        # Khởi tạo các chuỗi vị thế và giá entry
        self.pos = pd.Series(0, index=self.data.index)
        self.entry_price = pd.Series(np.nan, index=self.data.index)

    def run_strategy(self):
        """
        Chạy chiến lược giao dịch với logic mean reversion dựa trên các điều kiện của RSI, trend và Bollinger Bands.
        """
        rolling_std = self.returns.rolling(10).std()
        for i in range(1, len(self.close)):
            # Tránh giao dịch nếu biến động thấp hơn 0,1%
            if rolling_std.iloc[i] < 0.001:
              continue
            current_index = self.close.index[i]
            prev_index = self.close.index[i - 1]
            # print(f"Current Candle: {current_index}")
            # Nếu đang trong giai đoạn mean reversion: trend và neutral đều đúng
            if self.trend.loc[current_index] and self.neutral.loc[current_index]:
                # Kiểm tra điều kiện vào lệnh long:
                if self.data.loc[prev_index, 'Low'] * (1 - self.thresh_std.loc[prev_index]) < self.bbl.loc[current_index] and self.data.loc[prev_index, 'Low'] < self.bbl.loc[current_index] and np.isnan(self.entry_price.loc[prev_index]):
                  self.pos.iloc[i] = -1  # Mở vị thế long
                  self.entry_price.iloc[i] = self.close.iloc[i]
                  # print(f"Open Long: {self.entry_price.iloc[i]}, {self.data.loc[current_index, 'Low'] * (1 - self.thresh_std.loc[current_index])}, {self.bbl.loc[current_index]}")
                # Kiểm tra điều kiện vào lệnh short:
                elif self.data.loc[prev_index, 'High'] * (1 + self.thresh_std.loc[prev_index]) > self.bbu.loc[current_index] and self.data.loc[prev_index, 'High'] > self.bbu.loc[current_index] and np.isnan(self.entry_price.loc[prev_index]):
                  self.pos.iloc[i] = 1  # Mở vị thế short
                  self.entry_price.iloc[i] = self.close.iloc[i]
                  # print(f"Open Short: {self.entry_price.iloc[i]}, {self.data.loc[current_index, 'High'] * (1 + self.thresh_std.loc[current_index])}, {self.bbu.loc[current_index]}")
                else:
                  self.pos.iloc[i] = self.pos.iloc[i - 1]
                  self.entry_price.iloc[i] = self.entry_price.iloc[i - 1]

            else:
                # Nếu không thỏa điều kiện vào lệnh, giữ nguyên vị thế và giá entry từ phiên trước
                self.pos.iloc[i] = self.pos.iloc[i - 1]
                self.entry_price.iloc[i] = self.entry_price.iloc[i - 1]

            # Check for stop-loss and take-profit
            # --- Kiểm tra thoát lệnh cho vị thế Long ---
            if self.pos.iloc[i] == 1:
                # Nếu giá hiện tại đạt mức Take Profit (TP) cho lệnh Long
                if self.close.iloc[i] >= self.entry_price.iloc[i] * (1 + self.tp.iloc[i]):
                    # print(f"Current Candle: {current_index} - TP Long: Entry - {self.entry_price.iloc[i]}, TP - {self.close.iloc[i]}")
                    self.pos.iloc[i] = 0                  # Đóng vị thế
                    self.entry_price.iloc[i] = np.nan     # Reset giá vào lệnh
                # Nếu giá hiện tại chạm Stop Loss (SL) cho lệnh Long
                elif self.close.iloc[i] <= self.entry_price.iloc[i] * (1 + self.sl.iloc[i]):
                    # print(f"Current Candle: {current_index} - SL Long: Entry - {self.entry_price.iloc[i]}, SL - {self.close.iloc[i]}")
                    self.pos.iloc[i] = 0                  # Đóng vị thế
                    self.entry_price.iloc[i] = np.nan     # Reset giá vào lệnh

            # --- Kiểm tra thoát lệnh cho vị thế Short ---
            elif self.pos.iloc[i] == -1:
                # Nếu giá hiện tại đạt mức Take Profit (TP) cho lệnh Short
                if self.close.iloc[i] <= self.entry_price.iloc[i] * (1 - self.tp.iloc[i]):
                    # print(f"Current Candle: {current_index} - TP Short: Entry - {self.entry_price.iloc[i]}, TP - {self.close.iloc[i]}")
                    self.pos.iloc[i] = 0                  # Đóng vị thế
                    self.entry_price.iloc[i] = np.nan     # Reset giá vào lệnh
                # Nếu giá hiện tại chạm Stop Loss (SL) cho lệnh Short
                elif self.close.iloc[i] >= self.entry_price.iloc[i] * (1 - self.sl.iloc[i]):
                    # print(f"Current Candle: {current_index} - SL Short: Entry - {self.entry_price.iloc[i]}, SL - {self.close.iloc[i]}")
                    self.pos.iloc[i] = 0                  # Đóng vị thế
                    self.entry_price.iloc[i] = np.nan     # Reset giá vào lệnh

        # Reset các vị thế ban đầu nếu cần (ở đây reset 100 phiên đầu)
        self.pos.iloc[0:100] = 0

        # Tính toán Equity curve dựa trên lợi nhuận chưa hiện thực #SUA LAI CACH TINH RETURN
        self.unrlz_pnls = self.pos.shift(1) * self.returns * 0.96
        self.unrlz_pnls_cum = self.unrlz_pnls.cumsum() * 100

        # print("Số lượng lệnh được mở:", self.entry_price.notna().sum())
        # print("Số lượng phiên có vị thế long:", (self.pos == 1).sum())
        # print("Số lượng phiên có vị thế short:", (self.pos == -1).sum())
        # print("Số lượng phiên không có lệnh:", (self.pos == 0).sum())

    def plot_strategy_overlay(self):
      df = self.data.copy()

      df['Entry'] = self.entry_price
      df['TP'] = np.where(
          self.pos == 1,
          self.entry_price * (1 + self.tp),  # Buy: TP cao hơn entry
          self.entry_price * (1 - self.tp)   # Sell: TP thấp hơn entry
      )

      df['SL'] = np.where(
          self.pos == 1,
          self.entry_price * (1 + self.sl),  # Buy: SL cao hơn entry
          self.entry_price * (1 - self.sl)   # Sell: SL thấp hơn entry
      )

      df['BBU'] = self.bbu
      df['BBL'] = self.bbl
      df['LowOvershoot'] = df['Low'] * (1 - self.thresh_std)
      df['HighOvershoot'] = df['High'] * (1 + self.thresh_std)
      # for i in range(len(df)):
      #   print(df.loc[i, ['Entry','TP','SL']])
      fig = go.Figure()

      # Biểu đồ nến
      fig.add_trace(go.Candlestick(
          x=df['open_time'],
          open=df['Open'],
          high=df['High'],
          low=df['Low'],
          close=df['Close'],
          name='Candles'
      ))

      # Các đường bổ sung
      fig.add_trace(go.Scatter(x=df['open_time'], y=df['Entry'], name='Entry', line=dict(color='green')))
      fig.add_trace(go.Scatter(x=df['open_time'], y=df['TP'], name='Take Profit', line=dict(color='green', dash='dash')))
      fig.add_trace(go.Scatter(x=df['open_time'], y=df['SL'], name='Stop Loss', line=dict(color='red', dash='dash')))
      fig.add_trace(go.Scatter(x=df['open_time'], y=df['BBU'], name='Bollinger Upper', line=dict(color='white')))
      fig.add_trace(go.Scatter(x=df['open_time'], y=df['BBL'], name='Bollinger Lower', line=dict(color='white')))
      fig.add_trace(go.Scatter(x=df['open_time'], y=df['LowOvershoot'], name='Low Overshoot', line=dict(color='purple', dash='dot')))
      fig.add_trace(go.Scatter(x=df['open_time'], y=df['HighOvershoot'], name='High Overshoot', line=dict(color='orange', dash='dot')))


      fig.update_layout(
          title='Interactive Candle Chart with Strategy Overlay',
          xaxis_title='Time',
          yaxis_title='Price',
          xaxis_rangeslider_visible=False,
          template='plotly_dark',
          height=700
      )

      fig.show()

    def plot_performance(self):
      fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

      # Plot Position
      ax1.plot(self.data['open_time'], self.pos, label="Position", color='blue')
      ax1.set_ylabel("Position")
      ax1.legend()

      # Plot Equity Curve
      ax2.plot(self.data['open_time'], self.unrlz_pnls_cum, label="Equity Curve", color='green')
      ax2.set_xlabel("Time")
      ax2.set_ylabel("Equity (%)")
      ax2.legend()

      fig.autofmt_xdate()
      plt.tight_layout()
      plt.show()

    def print_summary(self):
      """
      In ra các chỉ số tóm tắt của chiến lược, bao gồm:
      - Sharpe Ratio và tổng lợi nhuận của cả ticker và chiến lược
      - Max Drawdown
      - Longest Drawdown Duration
      """
      # Sharpe và tổng returns
      periods_per_year = get_periods_per_year(data_frequency)
      risk_free_rate = 0.08  # annualized risk-free rate (8%)
      rf_per_period = risk_free_rate / periods_per_year

      # Adjusted Sharpe ratio with risk-free rate
      sharpe_returns = ((self.returns.mean() - rf_per_period) / self.returns.std()) * np.sqrt(periods_per_year)
      sum_returns = self.returns.sum() * 100

      sharpe_strategy = ((self.unrlz_pnls.mean() - rf_per_period) / self.unrlz_pnls.std()) * np.sqrt(periods_per_year)
      sum_pnls = self.unrlz_pnls.sum() * 100

      # Equity curve
      equity_curve = self.unrlz_pnls_cum

      # Tính drawdown
      peak = equity_curve.cummax()
      drawdown = equity_curve - peak
      max_drawdown = drawdown.min()

      # Tính thời gian dài nhất equity chưa phục hồi (longest drawdown duration)
      drawdown_duration = 0
      max_duration = 0
      for i in range(1, len(equity_curve)):
          if equity_curve[i] < peak[i]:
              drawdown_duration += 1
              max_duration = max(max_duration, drawdown_duration)
          else:
              drawdown_duration = 0
      max_duration = max_duration / periods_per_year * 365

      # Tính win/lose ratio cho Long và Short
      long_wins = 0
      long_losses = 0
      short_wins = 0
      short_losses = 0

      for i in range(1, len(self.pos)):
          # Khi một lệnh được đóng (tức là pos từ 1 -> 0 hoặc -1 -> 0)
          if self.pos.iloc[i - 1] == 1 and self.pos.iloc[i] == 0:
              pnl = (self.close.iloc[i] - self.entry_price.iloc[i - 1]) / self.entry_price.iloc[i - 1]
              if pnl > 0:
                  long_wins += 1
              else:
                  long_losses += 1
          elif self.pos.iloc[i - 1] == -1 and self.pos.iloc[i] == 0:
              pnl = (self.entry_price.iloc[i - 1] - self.close.iloc[i]) / self.entry_price.iloc[i - 1]
              if pnl > 0:
                  short_wins += 1
              else:
                  short_losses += 1

      total_long_trades = long_wins + long_losses
      total_short_trades = short_wins + short_losses

      long_win_ratio = long_wins / total_long_trades if total_long_trades > 0 else 0
      short_win_ratio = short_wins / total_short_trades if total_short_trades > 0 else 0
      win_ratio = (long_wins + short_wins) / (total_long_trades + total_short_trades) if (total_long_trades + total_short_trades) > 0 else 0

      # Tính lose streak dài nhất
      current_lose_streak = 0
      max_lose_streak = 0

      for i in range(1, len(self.pos)):
          if self.pos.iloc[i - 1] == 1 and self.pos.iloc[i] == 0:
              pnl = (self.close.iloc[i] - self.entry_price.iloc[i - 1]) / self.entry_price.iloc[i - 1]
              if pnl <= 0:
                  current_lose_streak += 1
                  max_lose_streak = max(max_lose_streak, current_lose_streak)
              else:
                  current_lose_streak = 0
          elif self.pos.iloc[i - 1] == -1 and self.pos.iloc[i] == 0:
              pnl = (self.entry_price.iloc[i - 1] - self.close.iloc[i]) / self.entry_price.iloc[i - 1]
              if pnl <= 0:
                  current_lose_streak += 1
                  max_lose_streak = max(max_lose_streak, current_lose_streak)
              else:
                  current_lose_streak = 0

      print(f'Sharpe Ratio Ticker: {sharpe_returns:.4f}')
      print(f'Sum Ticker Returns: {sum_returns:.4f}%')
      print('-' * 50)
      print(f'Sharpe Ratio Strategy: {sharpe_strategy:.4f}')
      print(f'Sum Strategy PnLs: {sum_pnls:.4f}%')
      print(f'Max Drawdown: {max_drawdown:.4f}%')
      print(f'Longest Drawdown Duration: {max_duration:.0f} day(s)')
      print(f'Win Ratio: {win_ratio:.2%} ({long_wins + short_wins}/{total_long_trades + total_short_trades})')
      print(f'Long Win Ratio: {long_win_ratio:.2%} ({long_wins}/{total_long_trades})')
      print(f'Short Win Ratio: {short_win_ratio:.2%} ({short_wins}/{total_short_trades})')
      print(f'Max Lose Streak: {max_lose_streak} trade(s)')

      summary = {
        'Ticker': ticker,
        'Sharpe_Ticker': sharpe_returns,
        'Total_Returns_%': sum_returns,
        'Sharpe_Strategy': sharpe_strategy,
        'Total_PnLs_%': sum_pnls,
        'Max_Drawdown_%': max_drawdown,
        'Longest_Drawdown_Duration_days': max_duration,
        'Win_Ratio': win_ratio,
        'Long_Win_Ratio': long_win_ratio,
        'Short_Win_Ratio': short_win_ratio,
        'Max_Lose_Streak': max_lose_streak,
        'Total_Trades': total_long_trades + total_short_trades
    }
      write_summary_to_csv(summary, file_path)

def write_summary_to_csv(summary_dict: dict, file_path: str):
    """
    Ghi thông tin summary vào file CSV.

    Args:
        summary_dict (dict): Dictionary chứa thông tin tổng hợp chiến lược.
        file_path (str): Đường dẫn đến file CSV lưu trữ.
    """
    df_summary = pd.DataFrame([summary_dict])

    if not os.path.exists(file_path):
        df_summary.to_csv(file_path, index=False)
    else:
        df_summary.to_csv(file_path, mode='a', header=False, index=False)

    print(f"✅ Summary for {summary_dict.get('Ticker', 'Unknown')} saved to {file_path}")

def get_periods_per_year(freq: str) -> int:
    if freq.endswith('m'):  # phút
        minutes = int(freq[:-1])
        return int(365 * 24 * 60 / minutes)
    elif freq.endswith('h'):  # giờ
        hours = int(freq[:-1])
        return int(365 * 24 / hours)
    elif freq.endswith('d'):  # ngày
        return 365
    elif freq.endswith('w'):  # tuần
        return 52
    else:
        raise ValueError(f"Không hỗ trợ định dạng data_frequency: {freq}")
# ----------------------
# Ví dụ sử dụng
# ----------------------
if __name__ == "__main__":
    # Tạo đối tượng chỉ báo và tính toán các chỉ báo cần thiết
    indicators = Indicators(data)
    indicators.compute_indicators()

    # Tạo đối tượng chiến lược, chạy chiến lược, và hiển thị kết quả
    strategy = MainStrategy(indicators)
    strategy.run_strategy()
    strategy.plot_strategy_overlay()
    strategy.plot_performance()
    strategy.print_summary()


Output hidden; open in https://colab.research.google.com to view.

In [ ]:
if __name__ == "__main__":

  start_time = time.time()

  data_frequency = '15m'
  # Lấy danh sách các ticker và lọc theo USDT
  all_tickers = get_list_all_trading_pairs()
  tickers = filter_usdt_tickers(all_tickers)

  for index, ticker in enumerate(tickers):
    ticker_start_time = time.time()
    print(f"🔄 Đang xử lý {ticker}... (Ticker {index + 1}/{len(tickers)})")

    date_start = find_first_data_date(ticker)

    if date_start is None:
      print(f"Bỏ qua {ticker} do không tìm thấy dữ liệu đầu tiên.")
      continue

    download_ticker(ticker, date_start.strftime("%Y-%m-%d"), "2025-03-01", data_frequency)
    crypto_data = process_csv_files(ticker, data_frequency)

    # Prepare data
    data = crypto_data.reset_index(drop=True)

    # Tạo đối tượng chỉ báo và tính toán các chỉ báo cần thiết
    indicators = Indicators(data)
    indicators.compute_indicators()

    # Tạo đối tượng chiến lược, chạy chiến lược, và hiển thị kết quả
    strategy = MainStrategy(indicators)
    strategy.run_strategy()
    strategy.print_summary()

    # Hiển thị thời gian xử lý và ước tính còn lại
    elapsed_time = time.time() - ticker_start_time
    total_elapsed = time.time() - start_time
    avg_ticker_time = total_elapsed / (index + 1)
    remaining_tickers = len(tickers) - index - 1
    estimated_time = avg_ticker_time * remaining_tickers
    print(f"✅ {ticker}: {format_time(elapsed_time)} | Đã xử lý: {format_time(total_elapsed)} | Còn lại: {format_time(estimated_time)} ({index + 1}/{len(tickers)})")


🔄 Đang xử lý BTCUSDT... (Ticker 1/188)
---> Found overall tickers: 574
---> Filter to asked tickers: 7
------> Tickers left: 2
Download full data for 2 tickers: 
---> Data will be saved here: /content/spot
---> Data Frequency: 15m
---> Start Date: 20170801
---> End Date: 20250301


Tickers:   0%|          | 0/2 [00:00<?, ?it/s]

monthly files to download: 0files [00:00, ?files/s]

daily files to download: 0files [00:00, ?files/s]

monthly files to download: 0files [00:00, ?files/s]

...

daily files to download:   0%|          | 0/1 [00:00<?, ?files/s]

monthly files to download: 0files [00:00, ?files/s]

daily files to download: 0files [00:00, ?files/s]

Tried to dump data for 2 tickers:
---> For ADAUSDT new data saved for: 83 months 1 days
---> For ADAUSD new data saved for: 0 months 0 days


KeyboardInterrupt: 